# Question 8

# 8. Three-Level Namespace Plan for Cyntexa

## Proposed Namespace Structure

| **Level**   | **Object**     | **Cyntexa's Implementation**                    | **Purpose**                                                               |
| ----------- | -------------- | ----------------------------------------------- | ------------------------------------------------------------------------- |
| **Level 1** | **Catalog**    | `cyntexa_dev`, `cyntexa_stage`, `cyntexa_prod`  | Isolation of environments, security boundaries, and lifecycle management. |
| **Level 2** | **Schema**     | `hr`, `it`, `finance`, `engineering`, `sales`   | Logical grouping by business domain or department.                        |
| **Level 3** | **Table/View** | `employees`, `payroll`, `support_tickets`, etc. | Actual datasets containing business records.                              |

## Namespace Blueprint

### 1. Catalogs (Environment Isolation)

* **`cyntexa_dev`**: Sandbox environment for developers to experiment, run ad-hoc queries, and test new pipelines without affecting higher environments.

* **`cyntexa_stage`**: Pre-production environment populated with sanitized/sampled production data to test integration, automated deployments (CI/CD), and validation routines.

* **`cyntexa_prod`**: Highly secure environment housing trusted enterprise datasets for reporting, BI dashboards, and ML applications.

### 2. Schemas (Business Domains)

Under every catalog, replicate identical schema structures for organizational consistency:

* **`hr`**: Human Resources data, such as employee details, attendance, and performance reviews.
* **`it`**: IT infrastructure and operations data, such as asset tracking, access logs, and ticket systems.
* **`finance`**: Financial and accounting data, such as revenue, expenses, budgets, and invoices.
* **`engineering`**: R&D, product development logs, deployment metrics, and codebase analytics.
* **`sales`**: Sales-related data, such as customers, orders, transactions, and sales performance.

### 3. Tables / Views (Data Assets)

Each schema contains the required tables and views for that business domain.

For example, the fully qualified name (FQN) for production payroll data would be:

```sql
cyntexa_prod.finance.payroll_records
```

## Key Justifications for the Architecture

### 1. Strict Data Governance & Access Control (RBAC)

Managing environments at the **Catalog level** allows administrators to apply Role-Based Access Controls (RBAC).

For example, junior developers can be given full read/write access to:

```text
cyntexa_dev.*
```

while access to sensitive production data can be restricted:

```text
cyntexa_prod.finance.*
```

This provides strong isolation between development and production environments.

### 2. Domain Ownership (Data Mesh Alignment)

Structuring schemas by business domain allows ownership and governance to be delegated to the appropriate teams.

For example:

* HR team → `*.hr.*`
* Finance team → `*.finance.*`
* IT team → `*.it.*`
* Engineering team → `*.engineering.*`
* Sales team → `*.sales.*`

This makes data ownership and access responsibilities easier to manage.

### 3. Promotion Pipeline Simplicity (CI/CD)

Data engineering code can be developed once and promoted across environments by parameterizing the catalog name.

For example:

```text
cyntexa_dev.finance.payroll_records
cyntexa_stage.finance.payroll_records
cyntexa_prod.finance.payroll_records
```

This keeps the namespace structure consistent across environments and simplifies CI/CD deployments.

## Summary

The proposed **Catalog → Schema → Table/View** structure provides clear separation of environments, logical organization by business domains, centralized governance, and easier CI/CD promotion. It also provides a scalable foundation for managing Cyntexa's data assets as the organization grows.


# Question 9


# 1. Objective

The objective of the data-masking strategy is to protect sensitive customer information while still allowing authorized users to access the data required for their business responsibilities.

# 2. Identify Sensitive Columns

We can have columns like - `customer_id`, `email`, `phone`, `address` and many more.

These all columns need to masked. For example (`email` -> `masked_email('adi***@***.com')`).

# 3. Define Role Tiers

## Tier 1 – Data Administrator (DA)

Tier 1 users are responsible for managing and governing enterprise data. They may require access to original sensitive data for administrative and security-related activities.

## Tier 2 – Data Engineer (DE)

Data Engineers are responsible for data ingestion, transformation, pipelines, and data quality across the Bronze, Silver, and Gold layers.

They may need access to sensitive data for:

- Pipeline troubleshooting
- Data transformation
- Data quality checks
- CDC processing
- Schema and table maintenance
- Investigating ingestion failures

However, Data Engineers should not automatically receive unrestricted access to PII simply because they work with the data platform.

Where possible, sensitive fields should remain masked:

- Email: `j***@gmail.com`
- Phone: `******3210`

Unmasked access can be provided only when it is required for a specific operational responsibility and approved through the appropriate access controls.

## Tier 3 – Data Analyst (DA)

Data Analysts use data primarily for business analysis, reporting, KPI calculations, and analytical workloads.

Typical activities include:

- Sales analysis
- Customer segmentation
- KPI calculations
- Reporting
- Trend analysis
- Business intelligence

Analysts generally do not require the actual customer email address or phone number.

## Tier 4 – Data Consumer / Viewer

Tier 4 users are consumers of prepared data, dashboards, reports, or Gold-layer datasets.

They have the most restricted access to sensitive information.

# Example Masking Functions

---

```sql
CREATE OR REPLACE FUNCTION governance.masks.email_mask(email STRING)

RETURNS STRING

RETURN CASE

    WHEN is_account_group_member('tier1_data_admin')
      OR is_account_group_member('tier2_support_lead')
    THEN email

    ELSE regexp_replace(
        email,
        '^([^@]{1})[^@]+(@.+)$',
        '$1***$2'
    )

END;
```


#4. Unity Catalog Enforcement Mechanism

Unity Catalog enforces data masking dynamically at query runtime without altering physical Delta storage:

- **Central Governance:** Masking functions are stored in a secure, centralized schema (e.g., `governance.masks`). Only Tier 1 Admins have `CREATE` or `MODIFY` permissions on this schema, preventing lower-tier users from tampering with masking logic.

- **Runtime Evaluation:** The `SET MASK` clause intercepts incoming queries and uses `is_account_group_member()` to check user roles dynamically on the fly, rendering raw or masked data based on active group privileges.

- **Single Source of Truth:** Unmasked raw data stays preserved in Delta storage. Data masking occurs entirely in-memory (RAM) during query execution, eliminating the need to maintain duplicate, anonymized datasets.

- **Least Privilege & Permission Hierarchy:** Unity Catalog operates on an explicit top-down privilege model (`USE CATALOG` → `USE SCHEMA` → `SELECT`). Users must hold container-level permissions to access tables. Once granted, Tier 3 and 4 users execute standard SQL queries and automatically receive masked results without modifying their code.

- **Audit & Lineage:** Every query execution against masked entities is recorded in `system.access.audit` logs for compliance. Column-level lineage tracks downstream transformations, ensuring governance policies carry forward into Gold-layer tables.

- **Row-Level Security:** Unity Catalog can also enforce row-level filtering when users should only be allowed to access specific rows. For example, users can be restricted to customers belonging to their assigned state or region. A row filter function returns `TRUE` for rows the user is allowed to see and `FALSE` for rows that should be filtered out.

# Question 10


In [0]:
-- 10. (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to
-- produce a 'top 5 customers by revenue per region' report.


-- select *  from samples.tpch.customer ;
-- select * from samples.tpch.orders  

-- desc table samples.tpch.region;
-- desc table samples.tpch.customer;     -- cust key , nation key
-- desc table samples.tpch.orders;        // order key  , cust key
-- desc table samples.tpch.supplier;   -- // supp key  , nation key 
-- desc table samples.tpch.lineitem;   // order key , supp key 
-- desc table samples.tpch.nation;    -- nation key , region key 

with  customers_total_spent_region_wise as
(
select   r.r_regionkey ,r.r_name,  c.c_custkey,c.c_name ,  sum(o.o_totalprice) as  total_spent 
from samples.tpch.customer c
inner join samples.tpch.orders o on c.c_custkey = o.o_custkey
inner join samples.tpch.nation n on c.c_nationkey = n.n_nationkey
inner join samples.tpch.region r on  n.n_regionkey = r.r_regionkey
group by r.r_regionkey , r.r_name, c.c_custkey ,c.c_name

)
,
customers_total_spent_region_wise_with_rank as
(
select * , dense_rank() over (partition by r_regionkey order by total_spent desc) as rnk  from customers_total_spent_region_wise
)

select * from customers_total_spent_region_wise_with_rank  where rnk <=5
;
